# Supplementary figure — Participant correlations

Physiological and facial-expression correlations with temperature are kept in one notebook but rendered and exported as separate figures.

In [58]:
%%capture
from pathlib import Path

if Path.cwd().name == "notebooks_altair":
    %cd ..

%load_ext autoreload
%autoreload 2

In [59]:
%%capture
import polars as pl

from src.data.database_manager import DatabaseManager
from src.plots.correlations import (
    calculate_correlations_per_trial,
    calculate_participant_stats,
)
from src.plots_altair import plot_participant_correlations, style_figure

In [60]:
database = DatabaseManager()
with database:
    explore_data = database.get_trials("Explore_Data", exclude_problematic=True)

explore_data = explore_data.rename(
    {
        "rating": "pain_rating",
        "pupil": "pupil_diameter",
    }
)
correlation_data = explore_data.filter(pl.col("normalized_timestamp") >= 20 * 1000)
physiological_targets = [
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
]
facial_targets = [
    "brow_furrow",
    "cheek_raise",
    "mouth_open",
    "upper_lip_raise",
    "nose_wrinkle",
]

In [61]:
physiological_trial_correlations = calculate_correlations_per_trial(
    correlation_data,
    "temperature",
    physiological_targets,
)
physiological_stats = calculate_participant_stats(
    physiological_trial_correlations,
    physiological_targets,
)
facial_trial_correlations = calculate_correlations_per_trial(
    correlation_data,
    "temperature",
    facial_targets,
)
facial_stats = calculate_participant_stats(
    facial_trial_correlations,
    facial_targets,
)

In [62]:
width = 800
height = 400
point_opacity = 0.9

physiological_chart = plot_participant_correlations(
    physiological_stats,
    physiological_targets,
    reference="temperature",
    signal_labels=None,
    signal_colors=None,
    width=width,
    height=height,
    y_domain=(-0.4, 1.0),
    point_size=58,
    point_opacity=point_opacity,
    error_bar_width=1.3,
    cap_size=8,
    zero_line_opacity=0.35,
    participant_group_padding=0.45,
    vertical_grid_opacity=0.35,
    legend_columns=5,
    title=None,
)
facial_chart = plot_participant_correlations(
    facial_stats,
    facial_targets,
    reference="temperature",
    signal_labels=None,
    signal_colors=None,
    width=width,
    height=height,
    y_domain=(-0.7, 1.0),
    point_size=58,
    point_opacity=point_opacity,
    error_bar_width=1.3,
    cap_size=8,
    zero_line_opacity=0.35,
    participant_group_padding=0.45,
    vertical_grid_opacity=0.35,
    legend_columns=5,
    title=None,
)

In [63]:
style_figure(physiological_chart)

alt.LayerChart(...)

In [64]:
style_figure(facial_chart)

alt.LayerChart(...)

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
figure_dir = Path(os.environ["FIGURE_DIR"])
figure_dir.mkdir(parents=True, exist_ok=True)
style_figure(physiological_chart).save(figure_dir / "correlations_with_temperature.svg")
style_figure(facial_chart).save(figure_dir / "correlations_with_temperature_face.svg")